<a href="https://colab.research.google.com/github/realshubhamraut/CDAC-DBDA-coursework/blob/main/06.big-data-technologies/assignments/assignment_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import os       #importing os to set environment variable
def install_java():
  !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null      #install openjdk
  os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"     #set environment variable
  !java -version       #check java version
install_java()

openjdk version "1.8.0_462"
OpenJDK Runtime Environment (build 1.8.0_462-8u462-ga~us1-0ubuntu2~22.04.2-b08)
OpenJDK 64-Bit Server VM (build 25.462-b08, mixed mode)


# PySpark Airlines Dataset Assignment

Dataset: `/data/airlines.csv`

## Sections:
1. **Questions 1-9**: DataFrame API
2. **Questions 10-19**: Spark SQL
3. **Questions 20-39**: Data Analysis

## Initialize Spark Session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, max, desc

# Create Spark Session
spark = SparkSession.builder \
    .appName("Airlines Dataset Analysis") \
    .getOrCreate()

# Load the dataset
df = spark.read.csv("/data/airlines.csv", header=True, inferSchema=True)

# Display schema and sample data
df.printSchema()
df.show(5)

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

---
# Part 1: DataFrame API (Questions 1-9)

### Q1. Calculate the average revenue per seat for each year and quarter

In [ ]:
# Q1: Average revenue per seat for each year and quarter
avg_revenue_per_year_quarter = df.groupBy("Year", "Quarter") \
    .agg(avg("Avg_rev_per_seat").alias("Avg_Revenue_Per_Seat")) \
    .orderBy("Year", "Quarter")

avg_revenue_per_year_quarter.show()

### Q2. Find the year and quarter with the highest average revenue per seat

In [ ]:
# Q2: Year and quarter with highest average revenue per seat
highest_avg_revenue = df.orderBy(desc("Avg_rev_per_seat")).limit(1)
highest_avg_revenue.show()

# Alternative: Using groupBy
highest_avg_revenue_alt = df.groupBy("Year", "Quarter") \
    .agg(avg("Avg_rev_per_seat").alias("Avg_Revenue_Per_Seat")) \
    .orderBy(desc("Avg_Revenue_Per_Seat")) \
    .limit(1)
highest_avg_revenue_alt.show()

### Q3. Calculate the total number of booked seats for each year and quarter

In [ ]:
# Q3: Total number of booked seats for each year and quarter
total_booked_seats = df.groupBy("Year", "Quarter") \
    .agg(sum("booked_seats").alias("Total_Booked_Seats")) \
    .orderBy("Year", "Quarter")

total_booked_seats.show()

### Q4. Determine the year and quarter with the highest total number of booked seats

In [ ]:
# Q4: Year and quarter with highest total number of booked seats
highest_booked_seats = df.groupBy("Year", "Quarter") \
    .agg(sum("booked_seats").alias("Total_Booked_Seats")) \
    .orderBy(desc("Total_Booked_Seats")) \
    .limit(1)

highest_booked_seats.show()

### Q5. Calculate the total revenue generated for each year and quarter

In [ ]:
# Q5: Total revenue for each year and quarter
# Revenue = Avg_rev_per_seat * booked_seats
df_with_revenue = df.withColumn("Total_Revenue", col("Avg_rev_per_seat") * col("booked_seats"))

total_revenue_per_year_quarter = df_with_revenue.groupBy("Year", "Quarter") \
    .agg(sum("Total_Revenue").alias("Total_Revenue")) \
    .orderBy("Year", "Quarter")

total_revenue_per_year_quarter.show()

### Q6. Identify the year and quarter with the highest total revenue

In [ ]:
# Q6: Year and quarter with highest total revenue
highest_revenue = df_with_revenue.groupBy("Year", "Quarter") \
    .agg(sum("Total_Revenue").alias("Total_Revenue")) \
    .orderBy(desc("Total_Revenue")) \
    .limit(1)

highest_revenue.show()

### Q7. Find the average revenue per seat across different years

In [ ]:
# Q7: Average revenue per seat across different years
avg_revenue_per_year = df.groupBy("Year") \
    .agg(avg("Avg_rev_per_seat").alias("Avg_Revenue_Per_Seat")) \
    .orderBy("Year")

avg_revenue_per_year.show()

### Q8. Determine the year with the highest average revenue per seat

In [ ]:
# Q8: Year with highest average revenue per seat
highest_avg_revenue_year = df.groupBy("Year") \
    .agg(avg("Avg_rev_per_seat").alias("Avg_Revenue_Per_Seat")) \
    .orderBy(desc("Avg_Revenue_Per_Seat")) \
    .limit(1)

highest_avg_revenue_year.show()

### Q9. Calculate the overall average revenue per seat for the entire dataset

In [ ]:
# Q9: Overall average revenue per seat for entire dataset
overall_avg_revenue = df.agg(avg("Avg_rev_per_seat").alias("Overall_Avg_Revenue_Per_Seat"))
overall_avg_revenue.show()

---
# Part 2: Spark SQL (Questions 10-19)

### Q10. Create a database and table to store the airline CSV data

In [ ]:
# Q10: Create database and table
# Create or use database
spark.sql("CREATE DATABASE IF NOT EXISTS airlines_db")
spark.sql("USE airlines_db")

# Register DataFrame as a temporary view/table
df.createOrReplaceTempView("airlines")

# Verify table creation
spark.sql("SHOW TABLES").show()
spark.sql("SELECT * FROM airlines LIMIT 5").show()

### Q11. Calculate the average revenue per seat for each year and quarter using SQL

In [ ]:
# Q11: Average revenue per seat for each year and quarter
query = """
SELECT Year, Quarter, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
spark.sql(query).show()

### Q12. Find the year and quarter with the highest average revenue per seat using SQL

In [ ]:
# Q12: Year and quarter with highest average revenue per seat
query = """
SELECT Year, Quarter, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Year, Quarter
ORDER BY Avg_Revenue_Per_Seat DESC
LIMIT 1
"""
spark.sql(query).show()

### Q13. Calculate the total number of booked seats for each year and quarter using SQL

In [ ]:
# Q13: Total number of booked seats for each year and quarter
query = """
SELECT Year, Quarter, SUM(booked_seats) AS Total_Booked_Seats
FROM airlines
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
spark.sql(query).show()

### Q14. Determine the year and quarter with the highest total number of booked seats using SQL

In [ ]:
# Q14: Year and quarter with highest total number of booked seats
query = """
SELECT Year, Quarter, SUM(booked_seats) AS Total_Booked_Seats
FROM airlines
GROUP BY Year, Quarter
ORDER BY Total_Booked_Seats DESC
LIMIT 1
"""
spark.sql(query).show()

### Q15. Calculate the total revenue generated for each year and quarter using SQL

In [ ]:
# Q15: Total revenue for each year and quarter
query = """
SELECT Year, Quarter,
       SUM(Avg_rev_per_seat * booked_seats) AS Total_Revenue
FROM airlines
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
spark.sql(query).show()

### Q16. Identify the year and quarter with the highest total revenue using SQL

In [ ]:
# Q16: Year and quarter with highest total revenue
query = """
SELECT Year, Quarter,
       SUM(Avg_rev_per_seat * booked_seats) AS Total_Revenue
FROM airlines
GROUP BY Year, Quarter
ORDER BY Total_Revenue DESC
LIMIT 1
"""
spark.sql(query).show()

### Q17. Find the average revenue per seat across different years using SQL

In [ ]:
# Q17: Average revenue per seat across different years
query = """
SELECT Year, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Year
ORDER BY Year
"""
spark.sql(query).show()

### Q18. Determine the year with the highest average revenue per seat using SQL

In [ ]:
# Q18: Year with highest average revenue per seat
query = """
SELECT Year, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Year
ORDER BY Avg_Revenue_Per_Seat DESC
LIMIT 1
"""
spark.sql(query).show()

### Q19. Calculate the overall average revenue per seat for the entire dataset using SQL

In [ ]:
# Q19: Overall average revenue per seat for entire dataset
query = """
SELECT AVG(Avg_rev_per_seat) AS Overall_Avg_Revenue_Per_Seat
FROM airlines
"""
spark.sql(query).show()

---
# Part 3: Data Analysis Questions (Questions 20-39)

### Q20-22: Average revenue per seat analysis by year and quarter

In [ ]:
# Q20: Average revenue per seat for each year and quarter
query = """
SELECT Year, Quarter, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
result_q20 = spark.sql(query)
result_q20.show(50)

# Q21: Year and quarter with highest average revenue per seat
query = """
SELECT Year, Quarter, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Year, Quarter
ORDER BY Avg_Revenue_Per_Seat DESC
LIMIT 1
"""
result_q21 = spark.sql(query)
print("\n--- Q21: Year and Quarter with Highest Average Revenue per Seat ---")
result_q21.show()

# Q22: Variation across quarters within each year
query = """
SELECT Year, Quarter, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
result_q22 = spark.sql(query)
print("\n--- Q22: Average Revenue Variation Across Quarters Within Years ---")
result_q22.show(50)

### Q23-25: Total booked seats analysis by year and quarter

In [ ]:
# Q23: Total booked seats for each year and quarter
query = """
SELECT Year, Quarter, SUM(booked_seats) AS Total_Booked_Seats
FROM airlines
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
result_q23 = spark.sql(query)
result_q23.show(50)

# Q24: Year and quarter with highest total booked seats
query = """
SELECT Year, Quarter, SUM(booked_seats) AS Total_Booked_Seats
FROM airlines
GROUP BY Year, Quarter
ORDER BY Total_Booked_Seats DESC
LIMIT 1
"""
result_q24 = spark.sql(query)
print("\n--- Q24: Year and Quarter with Highest Total Booked Seats ---")
result_q24.show()

# Q25: Variation across quarters within each year
query = """
SELECT Year, Quarter, SUM(booked_seats) AS Total_Booked_Seats
FROM airlines
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
result_q25 = spark.sql(query)
print("\n--- Q25: Booked Seats Variation Across Quarters Within Years ---")
result_q25.show(50)

### Q26-29: Total revenue analysis by year and quarter

In [ ]:
# Q26: Total revenue for each year and quarter
query = """
SELECT Year, Quarter,
       SUM(Avg_rev_per_seat * booked_seats) AS Total_Revenue
FROM airlines
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
result_q26 = spark.sql(query)
result_q26.show(50)

# Q27: Year and quarter with highest total revenue
query = """
SELECT Year, Quarter,
       SUM(Avg_rev_per_seat * booked_seats) AS Total_Revenue
FROM airlines
GROUP BY Year, Quarter
ORDER BY Total_Revenue DESC
LIMIT 1
"""
result_q27 = spark.sql(query)
print("\n--- Q27: Year and Quarter with Highest Total Revenue ---")
result_q27.show()

# Q28: Variation across quarters within each year
query = """
SELECT Year, Quarter,
       SUM(Avg_rev_per_seat * booked_seats) AS Total_Revenue
FROM airlines
GROUP BY Year, Quarter
ORDER BY Year, Quarter
"""
result_q28 = spark.sql(query)
print("\n--- Q28: Total Revenue Variation Across Quarters Within Years ---")
result_q28.show(50)

### Q29: Identify trends and patterns in revenue/booked seats over the years

In [ ]:
# Q29: Trends analysis - Yearly aggregations
query = """
SELECT Year,
       AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat,
       SUM(booked_seats) AS Total_Booked_Seats,
       SUM(Avg_rev_per_seat * booked_seats) AS Total_Revenue
FROM airlines
GROUP BY Year
ORDER BY Year
"""
result_q29 = spark.sql(query)
print("\n--- Q29: Trends and Patterns Over Years ---")
result_q29.show(50)

### Q30-32: Average revenue per seat across different years

In [ ]:
# Q30: Average revenue per seat across different years
query = """
SELECT Year, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Year
ORDER BY Year
"""
result_q30 = spark.sql(query)
print("\n--- Q30: Average Revenue Per Seat Across Years ---")
result_q30.show(50)

# Q31: Year with highest average revenue per seat
query = """
SELECT Year, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Year
ORDER BY Avg_Revenue_Per_Seat DESC
LIMIT 1
"""
result_q31 = spark.sql(query)
print("\n--- Q31: Year with Highest Average Revenue Per Seat ---")
result_q31.show()

# Q32: Overall average revenue per seat
query = """
SELECT AVG(Avg_rev_per_seat) AS Overall_Avg_Revenue_Per_Seat
FROM airlines
"""
result_q32 = spark.sql(query)
print("\n--- Q32: Overall Average Revenue Per Seat ---")
result_q32.show()

### Q33: Identify seasonal trends in average revenue per seat

In [ ]:
# Q33: Seasonal trends in average revenue per seat (by quarter)
query = """
SELECT Quarter, AVG(Avg_rev_per_seat) AS Avg_Revenue_Per_Seat
FROM airlines
GROUP BY Quarter
ORDER BY Quarter
"""
result_q33 = spark.sql(query)
print("\n--- Q33: Seasonal Trends in Average Revenue Per Seat ---")
result_q33.show()

### Q34-36: Total booked seats across different years

In [ ]:
# Q34: Total booked seats across different years
query = """
SELECT Year, SUM(booked_seats) AS Total_Booked_Seats
FROM airlines
GROUP BY Year
ORDER BY Year
"""
result_q34 = spark.sql(query)
print("\n--- Q34: Total Booked Seats Across Years ---")
result_q34.show(50)

# Q35: Year with highest total booked seats
query = """
SELECT Year, SUM(booked_seats) AS Total_Booked_Seats
FROM airlines
GROUP BY Year
ORDER BY Total_Booked_Seats DESC
LIMIT 1
"""
result_q35 = spark.sql(query)
print("\n--- Q35: Year with Highest Total Booked Seats ---")
result_q35.show()

# Q36: Overall total booked seats
query = """
SELECT SUM(booked_seats) AS Overall_Total_Booked_Seats
FROM airlines
"""
result_q36 = spark.sql(query)
print("\n--- Q36: Overall Total Booked Seats ---")
result_q36.show()

### Q37: Identify seasonal trends in total booked seats

In [ ]:
# Q37: Seasonal trends in total booked seats (by quarter)
query = """
SELECT Quarter, SUM(booked_seats) AS Total_Booked_Seats
FROM airlines
GROUP BY Quarter
ORDER BY Quarter
"""
result_q37 = spark.sql(query)
print("\n--- Q37: Seasonal Trends in Total Booked Seats ---")
result_q37.show()

### Q38-39: Total revenue across different years

In [ ]:
# Q38: Total revenue across different years
query = """
SELECT Year, SUM(Avg_rev_per_seat * booked_seats) AS Total_Revenue
FROM airlines
GROUP BY Year
ORDER BY Year
"""
result_q38 = spark.sql(query)
print("\n--- Q38: Total Revenue Across Years ---")
result_q38.show(50)

# Q39: Year with highest total revenue
query = """
SELECT Year, SUM(Avg_rev_per_seat * booked_seats) AS Total_Revenue
FROM airlines
GROUP BY Year
ORDER BY Total_Revenue DESC
LIMIT 1
"""
result_q39 = spark.sql(query)
print("\n--- Q39: Year with Highest Total Revenue ---")
result_q39.show()

---
## Summary and Insights

### Key Findings:
- Average revenue per seat trends by year and quarter
- Total booked seats patterns
- Total revenue calculations
- Seasonal trends identification
- Year-over-year comparisons